# 02. Preprocessing Pipeline & Feature Engineering
**Author**: Yordanos Andargachew (Phone: `+251 952 190 305`)  
**Project**: Bank Loan Risk Prediction System  

---

## 1. Objectives
This notebook constructs the production-grade data preprocessing pipeline:
1. **Handling Missing Values**: Impute missing categorical values in `Saving accounts` and `Checking account` with `'unknown'`.
2. **One-Hot Encoding**: Encode nominal categorical variables (`Sex`, `Housing`, `Saving accounts`, `Checking account`, `Purpose`).
3. **Feature Scaling**: Apply `StandardScaler` to continuous numerical features (`Age`, `Credit amount`, `Duration`).
4. **Stratified Train/Test Split**: 80% Train, 20% Test split (`random_state=42`).
5. **Serialization**: Fit and serialize `models/preprocessor.pkl`.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

# Add src to sys.path
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("."))

from src.preprocessor import CreditDataPreprocessor

# Load raw dataset
data_path = os.path.join("..", "data", "dataset.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "dataset.csv")

df = pd.read_csv(data_path)
print(f"Raw Data Loaded: {df.shape}")
df.head(5)

Raw Data Loaded: (1000, 10)


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,NaN,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad


## 2. Stratified 80/20 Train-Test Split

In [2]:
X = df.drop(columns=['Risk'])
y = df['Risk']

# Stratified split to preserve 70/30 class balance in train and test splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Set Shape: {X_train.shape} ({len(X_train)} samples)")
print(f"Test Set Shape:     {X_test.shape} ({len(X_test)} samples)")
print(f"Train Risk Balance: \n{y_train.value_counts(normalize=True).round(3)}")
print(f"Test Risk Balance:  \n{y_test.value_counts(normalize=True).round(3)}")

Training Set Shape: (800, 9) (800 samples)
Test Set Shape:     (200, 9) (200 samples)
Train Risk Balance: 
Risk
good    0.7
bad     0.3
Name: proportion, dtype: float64
Test Risk Balance:  
Risk
good    0.7
bad     0.3
Name: proportion, dtype: float64


## 3. Fitting the Preprocessing Pipeline

In [3]:
preprocessor = CreditDataPreprocessor()
preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Total Transformed Features: {len(feature_names)}")
print("Transformed Feature Names:")
for i, name in enumerate(feature_names):
    print(f"  {i+1:02d}. {name}")

Total Transformed Features: 26
Transformed Feature Names:
  01. Age
  02. Credit amount
  03. Duration
  04. Sex_female
  05. Sex_male
  06. Housing_free
  07. Housing_own
  08. Housing_rent
  09. Saving accounts_little
  10. Saving accounts_moderate
  11. Saving accounts_quite rich
  12. Saving accounts_rich
  13. Saving accounts_unknown
  14. Checking account_little
  15. Checking account_moderate
  16. Checking account_rich
  17. Checking account_unknown
  18. Purpose_business
  19. Purpose_car
  20. Purpose_domestic appliances
  21. Purpose_education
  22. Purpose_furniture/equipment
  23. Purpose_radio/TV
  24. Purpose_repairs
  25. Purpose_vacation/others
  26. Job


## 4. Validating Preprocessed Feature Matrix

In [4]:
df_transformed_sample = pd.DataFrame(X_train_transformed[:5], columns=feature_names)
print("Sample of Transformed Matrix:")
df_transformed_sample

Sample of Transformed Matrix:


,Age,Credit amount,Duration,Sex_female,Sex_male,Housing_free,Housing_own,Housing_rent,Saving accounts_little,Saving accounts_moderate,...,Checking account_unknown,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others,Job
0,-0.825479,0.485384,0.755149,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
1,0.493705,-0.246578,0.755149,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
2,-1.177262,-0.584573,-0.726746,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
3,-0.033969,0.285331,0.014201,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0
4,-1.177262,-0.319522,-0.973728,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0


## 5. Serializing Preprocessor Artifact

In [5]:
models_dir = os.path.join("..", "models")
if not os.path.exists(models_dir):
    models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

preprocessor_path = os.path.join(models_dir, "preprocessor.pkl")
preprocessor.save(preprocessor_path)
print(f"Successfully saved fitted preprocessor to '{preprocessor_path}'!")

# Test loading
loaded_prep = CreditDataPreprocessor.load(preprocessor_path)
test_out = loaded_prep.transform(X_test.head(1))
print(f"Verification Load Successful! Transformed single test row with shape {test_out.shape}.")

Successfully saved fitted preprocessor to '../models/preprocessor.pkl'!
Verification Load Successful! Transformed single test row with shape (1, 26).
